# ЛР-7. Многослойный персептрон (MLP)

Датасет: UCI Mushroom (id=73). Классификация грибов: съедобные / ядовитые.  
Архитектура: 2 скрытых слоя, сигмоида, ванильный градиентный спуск.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from mlcore.utils.notebook import lab_paths, setup_plotting
from mlcore.metrics.report import print_classification_metrics, save_confusion_matrix, save_binary_roc, save_loss_curve
from mlcore.data.datasets import load_lr7_dataset
from mlcore.preprocessing.encoding import label_encode, one_hot_encode
from mlcore.preprocessing.splitting import train_test_split
from mlcore.utils.timing import timer

ROOT, _, ASSETS = lab_paths(7)
setup_plotting()

## 1. Загрузка и анализ данных

In [ ]:
ds = load_lr7_dataset()
X_raw, y_raw = ds.features, ds.targets
print(f'Features: {X_raw.shape}, Target: {y_raw.shape}')
print(f'\nClass balance:\n{y_raw.value_counts()}')
print(f'\nMissing per column:')
# UCI Mushroom uses '?' for missing in stalk-root
print((X_raw == '?').sum().loc[lambda s: s > 0])
X_raw.head()

## 2. Предобработка

- Пропущенные значения `?` в `stalk-root` обрабатываем как отдельную категорию.
- One-hot кодирование всех 22 категориальных признаков.
- Целевая переменная: `p` (poisonous) = 1, `e` (edible) = 0.

In [ ]:
# One-hot encode all feature columns
encoded_parts = []
for col in X_raw.columns:
    oh, cats = one_hot_encode(X_raw[col].values)
    encoded_parts.append(oh)

X = np.hstack(encoded_parts)
y_enc, y_map = label_encode(y_raw.values.ravel())
# Ensure 'p' (poisonous) = 1
if y_map.get('e', -1) == 1:
    y_enc = 1 - y_enc
y = y_enc.astype(np.float64)

print(f'Encoded features: {X.shape}')
print(f'Target: 0=edible ({(y==0).sum()}), 1=poisonous ({(y==1).sum()}).')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 3. Реализация MLP

- 2 скрытых слоя (64, 32 нейрона), сигмоидная активация
- 1 выходной нейрон (сигмоида) — P(poisonous)
- Функция потерь: binary cross-entropy
- Оптимизация: ванильный градиентный спуск (full-batch)
- Инициализация весов: Xavier

In [ ]:
class MLP:
    """2-hidden-layer perceptron with sigmoid activation and vanilla GD."""

    def __init__(self, layer_sizes: list[int], random_state: int = 42):
        self.layer_sizes = layer_sizes  # [n_in, h1, h2, n_out]
        self.rng = np.random.default_rng(random_state)
        self.weights: list[np.ndarray] = []
        self.biases: list[np.ndarray] = []
        self.loss_history: list[float] = []
        self._init_weights()

    def _init_weights(self):
        """Xavier initialization."""
        for i in range(len(self.layer_sizes) - 1):
            fan_in, fan_out = self.layer_sizes[i], self.layer_sizes[i + 1]
            std = np.sqrt(2.0 / (fan_in + fan_out))
            self.weights.append(self.rng.normal(0, std, (fan_in, fan_out)))
            self.biases.append(np.zeros((1, fan_out)))

    @staticmethod
    def _sigmoid(z: np.ndarray) -> np.ndarray:
        return np.where(z >= 0, 1 / (1 + np.exp(-z)), np.exp(z) / (1 + np.exp(z)))

    def _forward(self, X: np.ndarray) -> tuple[list[np.ndarray], list[np.ndarray]]:
        activations = [X]
        zs = []
        a = X
        for W, b in zip(self.weights, self.biases):
            z = a @ W + b
            a = self._sigmoid(z)
            zs.append(z)
            activations.append(a)
        return activations, zs

    def _backward(self, y: np.ndarray, activations: list[np.ndarray]) -> tuple[list, list]:
        m = len(y)
        y_col = y.reshape(-1, 1)
        dW_list, db_list = [], []

        # Output layer: d(BCE)/d(z) = a - y (sigmoid + BCE simplification)
        delta = activations[-1] - y_col

        for i in range(len(self.weights) - 1, -1, -1):
            dW = activations[i].T @ delta / m
            db = delta.mean(axis=0, keepdims=True)
            dW_list.insert(0, dW)
            db_list.insert(0, db)
            if i > 0:
                a = activations[i]
                delta = (delta @ self.weights[i].T) * (a * (1 - a))  # sigmoid derivative

        return dW_list, db_list

    def fit(self, X: np.ndarray, y: np.ndarray, epochs: int = 1000,
            lr: float = 0.5, verbose: bool = True) -> 'MLP':
        self.loss_history = []
        eps = 1e-12
        for epoch in range(epochs):
            activations, _ = self._forward(X)
            y_hat = activations[-1]

            loss = -np.mean(y * np.log(y_hat + eps) + (1 - y) * np.log(1 - y_hat + eps))
            self.loss_history.append(loss)

            dW_list, db_list = self._backward(y, activations)
            for i in range(len(self.weights)):
                self.weights[i] -= lr * dW_list[i]
                self.biases[i] -= lr * db_list[i]

            if verbose and epoch % 200 == 0:
                print(f'  Epoch {epoch:4d}: loss={loss:.4f}')

        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        activations, _ = self._forward(X)
        return activations[-1].ravel()

    def predict(self, X: np.ndarray, threshold: float = 0.5) -> np.ndarray:
        return (self.predict_proba(X) >= threshold).astype(int)

## 4. Обучение

In [ ]:
n_features = X_train.shape[1]
model = MLP([n_features, 64, 32, 1], random_state=42)

with timer('MLP training'):
    model.fit(X_train, y_train, epochs=1000, lr=0.5)

In [ ]:
save_loss_curve(model.loss_history, ASSETS / 'training_loss.png', ylabel='BCE Loss')

## 5. Оценка качества

In [ ]:
y_pred = model.predict(X_test)

print_classification_metrics(y_test, y_pred)

In [ ]:
save_confusion_matrix(y_test, y_pred, ['edible', 'poisonous'], ASSETS / 'confusion_matrix.png')

In [ ]:
y_proba = model.predict_proba(X_test)
save_binary_roc(y_test, y_proba, ASSETS / 'roc_curve.png')

## 6. Выводы

1. MLP с архитектурой [input, 64, 32, 1] и сигмоидной активацией успешно классифицирует грибы.
2. Xavier инициализация и full-batch GD обеспечивают стабильную сходимость.
3. One-hot кодирование всех 22 категориальных признаков (~117 входов) достаточно для высокого качества.
4. Пропущенные значения в `stalk-root` обработаны как отдельная категория.